# RetailIQ 360° — Fase 1: Generación de datos sintéticos con Faker

**Objetivo:** Crear la capa de localización argentina que no existe en Olist ni Superstore.

**Archivos que genera este notebook:**
- `dim_geografia_ar.csv` — provincias y ciudades de Argentina
- `dim_sucursales_ar.csv` — 50 sucursales ficticias distribuidas por región
- `dim_canal_ar.csv` — canales de venta (físico, web, app, marketplace)
- `dim_clientes_ar.csv` — 10.000 clientes ficticios con perfil argentino
- `fact_ventas_ar.csv` — transacciones con columnas argentinas (medio_pago, cuotas, entrega)
- `fact_precios_comp.csv` — precios de competencia simulados por SKU

**Proporciones calibradas con datos reales de CACE 2025**

---
> **Antes de correr:** Abrí una terminal en VS Code y ejecutá:
> ```
> pip install faker pandas numpy
> ```

## BLOQUE 0 — Instalación y verificación
Corré este bloque primero. Si no da error, todo está listo.

In [1]:
# Instalación (solo la primera vez)
# Si ya instalaste desde la terminal, podés saltear esta celda
import subprocess
subprocess.run(['pip', 'install', 'faker', 'pandas', 'numpy', '--quiet'])

# Verificación
import faker
import pandas as pd
import numpy as np
print(f'✓ Faker versión: {faker.VERSION}')
print(f'✓ Pandas versión: {pd.__version__}')
print('✓ Todo listo para generar datos')

✓ Faker versión: 40.15.0
✓ Pandas versión: 3.0.2
✓ Todo listo para generar datos


## BLOQUE 1 — Importaciones y configuración inicial

Acá configuramos Faker con locale argentino y definimos la semilla aleatoria.

**¿Qué es `random_state` / `seed`?**  
Asegura que cada vez que corras el notebook obtengas los mismos datos.  
Sin seed, los datos cambiarían cada vez — eso rompería las relaciones entre tablas.

In [5]:
from faker import Faker
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

# ── Configuración global ─────────────────────────────────────
SEED = 42          # semilla fija → mismos datos siempre
random.seed(SEED)
np.random.seed(SEED)

# Faker con locale argentino
# es_AR genera: nombres, apellidos y formatos argentinos
fake = Faker('es_AR')
Faker.seed(SEED)

# Carpeta de salida (creala si no existe)
OUTPUT_DIR = '../datos/03_sinteticos'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Configuración lista')
print(f'Carpeta de salida: {os.path.abspath(OUTPUT_DIR)}')

# --- Pequeña demostración de Faker ---
print('\nEjemplos de lo que genera Faker con locale es_AR:')
print(f'  Nombre:    {fake.name()}')
print(f'  Email:     {fake.email()}')
print(f'  Teléfono:  {fake.phone_number()}')
print(f'  Empresa:   {fake.company()}')
print(f'  Fecha:     {fake.date_between(start_date="-2y", end_date="today")}')

Configuración lista
Carpeta de salida: c:\Proyectos\integrador_carrera\datos\03_sinteticos

Ejemplos de lo que genera Faker con locale es_AR:
  Nombre:    Benjamin Perez Diaz
  Email:     peraltauma@example.org
  Teléfono:  +54 9 3100 1338
  Empresa:   Diaz, Ferreyra and Molina
  Fecha:     2024-10-21


## BLOQUE 2 — Datos maestros de Argentina

Definimos las provincias, ciudades y regiones con los **pesos de facturación reales de CACE 2025**.  
Estos pesos determinan qué tan frecuente es cada región en las transacciones generadas.

**Fuente:** `cace_06a_distribucion_regional.csv` — AMBA 52%, Litoral 12%, Sur 9%, etc.

In [6]:
# ── Provincias con ciudades y pesos de facturación CACE 2025 ──
PROVINCIAS_AR = {
    # region: [(provincia, [ciudades], peso_facturacion_cace_2025)
    'AMBA': [
        ('Ciudad Autónoma de Buenos Aires', ['CABA', 'Palermo', 'Belgrano', 'Caballito', 'Flores'], 0.25),
        ('Buenos Aires GBA Norte',          ['San Isidro', 'Vicente López', 'Tigre', 'Pilar'], 0.12),
        ('Buenos Aires GBA Oeste',          ['Morón', 'La Matanza', 'Merlo', 'Moreno'], 0.10),
        ('Buenos Aires GBA Sur',            ['Quilmes', 'Lomas de Zamora', 'Lanús', 'Avellaneda'], 0.05),
    ],
    'Litoral': [
        ('Santa Fe',    ['Rosario', 'Santa Fe Capital', 'Rafaela', 'Venado Tuerto'], 0.06),
        ('Entre Ríos',  ['Paraná', 'Concordia', 'Gualeguaychú'], 0.03),
        ('Corrientes',  ['Corrientes Capital', 'Goya'], 0.02),
        ('Misiones',    ['Posadas', 'Eldorado', 'Oberá'], 0.01),
    ],
    'Centro': [
        ('Córdoba', ['Córdoba Capital', 'Villa Carlos Paz', 'Río Cuarto', 'Bell Ville'], 0.09),
    ],
    'Sur': [
        ('Neuquén',       ['Neuquén Capital', 'San Martín de los Andes', 'Bariloche'], 0.03),
        ('Río Negro',     ['Viedma', 'Bariloche', 'Cipolletti'], 0.02),
        ('Chubut',        ['Rawson', 'Comodoro Rivadavia', 'Puerto Madryn'], 0.02),
        ('Santa Cruz',    ['Río Gallegos', 'Caleta Olivia'], 0.01),
        ('Tierra del Fuego', ['Ushuaia', 'Río Grande'], 0.01),
    ],
    'NOA': [
        ('Tucumán',           ['San Miguel de Tucumán', 'Yerba Buena', 'Tafí Viejo'], 0.03),
        ('Salta',             ['Salta Capital', 'Orán', 'Tartagal'], 0.02),
        ('Jujuy',             ['San Salvador de Jujuy', 'Palpalá'], 0.01),
        ('Santiago del Estero', ['Santiago del Estero Capital', 'La Banda'], 0.005),
        ('Catamarca',         ['San Fernando del Valle de Catamarca'], 0.003),
        ('La Rioja',          ['La Rioja Capital'], 0.002),
    ],
    'Cuyo': [
        ('Mendoza',   ['Mendoza Capital', 'Godoy Cruz', 'Luján de Cuyo', 'San Rafael'], 0.03),
        ('San Juan',  ['San Juan Capital', 'Rawson', 'Rivadavia'], 0.01),
        ('San Luis',  ['San Luis Capital', 'Villa Mercedes'], 0.01),
    ],
}

# Aplanar para uso fácil
todas_las_provincias = []
for region, lista in PROVINCIAS_AR.items():
    for prov, ciudades, peso in lista:
        todas_las_provincias.append({
            'region': region,
            'provincia': prov,
            'ciudades': ciudades,
            'peso': peso
        })

print(f'Total de provincias/zonas definidas: {len(todas_las_provincias)}')
print(f'Suma de pesos (debe ser ~1.0): {sum(p["peso"] for p in todas_las_provincias):.3f}')

Total de provincias/zonas definidas: 23
Suma de pesos (debe ser ~1.0): 0.940


## BLOQUE 3 — DimGeografia

Tabla de dimensión geográfica. Una fila por combinación provincia-ciudad.  
Esta tabla se relaciona con `DimSucursal` y con `DimCliente` en el modelo de Power BI.

In [9]:
filas_geo = []
geo_id = 1

for entrada in todas_las_provincias:
    for ciudad in entrada['ciudades']:
        filas_geo.append({
            'GeografiaID':      geo_id,
            'region':           entrada['region'],
            'provincia':        entrada['provincia'],
            'ciudad':           ciudad,
            'zona':             'AMBA' if entrada['region'] == 'AMBA' else 'Interior',
            'peso_facturacion': entrada['peso'],   # peso CACE 2025 — para calibrar ventas
        })
        geo_id += 1

dim_geografia = pd.DataFrame(filas_geo)
dim_geografia.to_csv(f'{OUTPUT_DIR}/001_dim_geografia_ar.csv', index=False, encoding='utf-8-sig')

print(f'DimGeografia generada: {len(dim_geografia)} filas')
print(f'\nPrimeras 5 filas:')
dim_geografia.head()

DimGeografia generada: 67 filas

Primeras 5 filas:


,GeografiaID,region,provincia,ciudad,zona,peso_facturacion
0,1,AMBA,Ciudad Autónoma de Buenos Aires,CABA,AMBA,0.25
1,2,AMBA,Ciudad Autónoma de Buenos Aires,Palermo,AMBA,0.25
2,3,AMBA,Ciudad Autónoma de Buenos Aires,Belgrano,AMBA,0.25
3,4,AMBA,Ciudad Autónoma de Buenos Aires,Caballito,AMBA,0.25
4,5,AMBA,Ciudad Autónoma de Buenos Aires,Flores,AMBA,0.25


## BLOQUE 4 — DimSucursal

50 sucursales ficticias distribuidas por región con pesos realistas.  
Cada sucursal tiene tipo (física grande, física chica, dark store) y fecha de apertura.

**Faker en acción:** `fake.company()` genera nombres de empresas en español.

In [10]:
N_SUCURSALES = 50

# Distribución de sucursales por región (no igual a facturación — AMBA tiene más tiendas)
pesos_sucursales = {
    'AMBA':    0.50,   # 25 sucursales en AMBA
    'Litoral': 0.16,   # 8 en Litoral
    'Centro':  0.12,   # 6 en Córdoba
    'Sur':     0.10,   # 5 en el Sur
    'NOA':     0.07,   # 3-4 en NOA
    'Cuyo':    0.05,   # 2-3 en Cuyo
}

tipos_sucursal = ['Tienda grande', 'Tienda chica', 'Dark store', 'Centro de distribución']
pesos_tipo     = [0.40, 0.35, 0.15, 0.10]

filas_suc = []
for i in range(1, N_SUCURSALES + 1):
    
    # Elegir región con peso
    region = random.choices(
        list(pesos_sucursales.keys()),
        weights=list(pesos_sucursales.values())
    )[0]
    
    # Elegir provincia dentro de esa región
    opciones_region = [p for p in todas_las_provincias if p['region'] == region]
    entrada = random.choice(opciones_region)
    ciudad  = random.choice(entrada['ciudades'])
    
    # GeografiaID correspondiente
    fila_geo = dim_geografia[
        (dim_geografia['provincia'] == entrada['provincia']) &
        (dim_geografia['ciudad'] == ciudad)
    ]
    geo_id_suc = int(fila_geo['GeografiaID'].values[0]) if len(fila_geo) > 0 else 1
    
    tipo = random.choices(tipos_sucursal, weights=pesos_tipo)[0]
    
    # m2 varía según tipo
    m2_por_tipo = {
        'Tienda grande':         random.randint(800, 3000),
        'Tienda chica':          random.randint(100, 400),
        'Dark store':            random.randint(200, 600),
        'Centro de distribución':random.randint(1000, 5000),
    }
    
    filas_suc.append({
        'SucursalID':     i,
        'nombre':         f'RetailIQ {ciudad} {i}',
        'tipo':           tipo,
        'GeografiaID':    geo_id_suc,
        'provincia':      entrada['provincia'],
        'ciudad':         ciudad,
        'region':         region,
        'm2_ventas':      m2_por_tipo[tipo],
        'empleados':      random.randint(5, 120),
        'fecha_apertura': fake.date_between(start_date='-8y', end_date='-6m'),
        'flag_activa':    random.choices([1, 0], weights=[0.92, 0.08])[0],
    })

dim_sucursales = pd.DataFrame(filas_suc)
dim_sucursales.to_csv(f'{OUTPUT_DIR}/002_dim_sucursales_ar.csv', index=False, encoding='utf-8-sig')

print(f'DimSucursal generada: {len(dim_sucursales)} filas')
print(f'\nDistribución por región:')
print(dim_sucursales['region'].value_counts())
print(f'\nDistribución por tipo:')
print(dim_sucursales['tipo'].value_counts())

DimSucursal generada: 50 filas

Distribución por región:
region
AMBA       28
Litoral    10
Centro      5
Sur         5
NOA         1
Cuyo        1
Name: count, dtype: int64

Distribución por tipo:
tipo
Tienda chica              18
Tienda grande             15
Centro de distribución     9
Dark store                 8
Name: count, dtype: int64


## BLOQUE 5 — DimCanal

Tabla pequeña pero crítica: define los 4 canales del modelo omnicanal.  
Los pesos de facturación vienen de CACE 2025 (eCommerce propio 45%, marketplace 47%, físico 8%).

**Nota:** `flag_online` es lo que usamos en la medida DAX `Part. Online %`.

In [11]:
dim_canal = pd.DataFrame([
    # CanalID, nombre, tipo, plataforma, flag_online, peso_facturacion_cace_2025
    {'CanalID': 1, 'nombre': 'Tienda física',   'tipo': 'Físico',      'plataforma': 'POS físico',    'flag_online': 0, 'peso_facturacion': 0.08},
    {'CanalID': 2, 'nombre': 'Web propia',       'tipo': 'Online',      'plataforma': 'Sitio web',     'flag_online': 1, 'peso_facturacion': 0.25},
    {'CanalID': 3, 'nombre': 'App móvil',        'tipo': 'Online',      'plataforma': 'App iOS/Android','flag_online': 1, 'peso_facturacion': 0.20},
    {'CanalID': 4, 'nombre': 'Marketplace',      'tipo': 'Marketplace', 'plataforma': 'MercadoLibre',  'flag_online': 1, 'peso_facturacion': 0.47},
])

dim_canal.to_csv(f'{OUTPUT_DIR}/003_dim_canal_ar.csv', index=False, encoding='utf-8-sig')

print('DimCanal generada:')
print(dim_canal.to_string(index=False))

DimCanal generada:
 CanalID        nombre        tipo      plataforma  flag_online  peso_facturacion
       1 Tienda física      Físico      POS físico            0              0.08
       2    Web propia      Online       Sitio web            1              0.25
       3     App móvil      Online App iOS/Android            1              0.20
       4   Marketplace Marketplace    MercadoLibre            1              0.47


## BLOQUE 6 — DimCliente

10.000 clientes ficticios con perfil sociodemográfico argentino calibrado con CACE 2025.

**Faker genera:** nombre, email, teléfono.  
**Nosotras controlamos:** provincia (con pesos CACE), NSE, edad, canal preferido.

**¿Qué es `np.random.choice` con `p=`?**  
Elige aleatoriamente de una lista pero respetando las probabilidades que vos definís.  
Si ponés `p=[0.24, 0.26, 0.30, 0.20]` para NSE, el 24% de los clientes será ABC1, el 26% C2, etc. — igual que en el mercado real según CACE.

In [12]:
N_CLIENTES = 10_000

# Proporciones de provincia (pesos de DimGeografia, normalizados)
pesos_prov = [p['peso'] for p in todas_las_provincias]
suma_pesos = sum(pesos_prov)
pesos_prov_norm = [p / suma_pesos for p in pesos_prov]   # normalizar a 1.0

# NSE según CACE Anual 2025 p.5
nse_opciones = ['ABC1', 'C2', 'C3', 'D']
nse_pesos    = [0.24,   0.26,  0.30,  0.20]   # fuente: CACE 2025

# Edad según CACE 2025
edad_rangos = ['18-20', '21-29', '30-34', '35-44', '45-59', '60+']
edad_pesos  = [0.05,    0.26,    0.20,    0.28,    0.17,    0.04]

# Canal preferido (refleja que el 92% compra online)
canal_pref_opts  = ['Online', 'Físico', 'Indistinto']
canal_pref_pesos = [0.60,     0.15,     0.25]

filas_cli = []
for i in range(1, N_CLIENTES + 1):
    
    # Elegir provincia con pesos CACE
    idx_prov  = np.random.choice(len(todas_las_provincias), p=pesos_prov_norm)
    entrada   = todas_las_provincias[idx_prov]
    ciudad    = random.choice(entrada['ciudades'])
    
    filas_cli.append({
        'ClienteID':       i,
        'nombre':          fake.name(),          # Faker genera nombre argentino
        'email':           fake.email(),
        'telefono':        fake.phone_number(),
        'provincia':       entrada['provincia'],
        'ciudad':          ciudad,
        'region':          entrada['region'],
        'zona':            'AMBA' if entrada['region'] == 'AMBA' else 'Interior',
        'nse':             np.random.choice(nse_opciones, p=nse_pesos),
        'rango_edad':      np.random.choice(edad_rangos, p=edad_pesos),
        'genero':          np.random.choice(['M', 'F'], p=[0.50, 0.50]),
        'canal_preferido': np.random.choice(canal_pref_opts, p=canal_pref_pesos),
        'segmento_rfm':    None,   # se calcula después con DAX
        'fecha_alta':      fake.date_between(start_date='-5y', end_date='-1m'),
    })

dim_clientes = pd.DataFrame(filas_cli)
dim_clientes.to_csv(f'{OUTPUT_DIR}/004_dim_clientes_ar.csv', index=False, encoding='utf-8-sig')

print(f'DimCliente generada: {len(dim_clientes):,} filas')
print('\nDistribución NSE (debe aproximarse a CACE 2025):')
print(dim_clientes['nse'].value_counts(normalize=True).round(2))
print('\nDistribución por región:')
print(dim_clientes['region'].value_counts(normalize=True).round(2))

DimCliente generada: 10,000 filas

Distribución NSE (debe aproximarse a CACE 2025):
nse
C3      0.30
C2      0.26
ABC1    0.25
D       0.19
Name: proportion, dtype: float64

Distribución por región:
region
AMBA       0.55
Litoral    0.13
Sur        0.10
Centro     0.09
NOA        0.08
Cuyo       0.05
Name: proportion, dtype: float64


## BLOQUE 7 — FactVentas con columnas argentinas

Este es el bloque más importante. Genera 150.000 transacciones con:
- Medio de pago con proporciones CACE 2025
- Tipo de entrega con proporciones CACE 2025  
- Plazo de entrega diferenciado AMBA vs Interior
- Cuotas con distribución CACE 2025
- Estacionalidad argentina (Hot Sale en mayo, CyberMonday en octubre)

**Importante:** Este bloque genera solo las columnas "argentinas" nuevas.  
Las columnas de ventas (monto, producto, etc.) vienen de Olist/Superstore — las integramos en el siguiente notebook de ETL.

In [13]:
N_VENTAS = 150_000

# ── Proporciones CACE 2025 ────────────────────────────────────

# Medios de pago — fuente: cace_04a_medios_pago_oferta.csv (Anual 2025)
medios_pago  = ['tarjeta_credito_plataforma', 'billetera_electronica', 'debito_online',
                'tarjeta_credito_gateway',    'transferencia_bancaria', 'efectivo_rapipago',
                'debito_al_retirar',          'credito_al_retirar']
pesos_pago   = [0.55, 0.24, 0.09, 0.05, 0.03, 0.02, 0.01, 0.01]

# Tipo de entrega — fuente: cace_05a_logistica_tipo_entrega.csv (Anual 2025)
tipos_entrega = ['envio_domicilio', 'retiro_punto_venta', 'retiro_operador_logistico',
                 'retiro_pickup',    'mensajeria_rapida']
pesos_entrega = [0.60, 0.29, 0.08, 0.02, 0.01]

# Plazo de entrega por zona — fuente: cace_05b_plazos_entrega.csv
plazos        = ['same_day', '24hs', '48hs', 'en_semana', '15_dias', 'mas_de_mes']
pesos_AMBA    = [0.22, 0.17, 0.15, 0.34, 0.05, 0.01, 0.00, 0.00]  # 6 valores
pesos_AMBA    = [0.22, 0.17, 0.15, 0.34, 0.05, 0.07]
pesos_interior= [0.13, 0.14, 0.11, 0.45, 0.12, 0.05]

# Cuotas — fuente: cace_04b_financiamiento_cuotas.csv (2025)
cuotas_opts  = [1,    3,    6,    9,    12,   18]
pesos_cuotas = [0.27, 0.28, 0.26, 0.08, 0.06, 0.05]

# Canal — fuente: cace_01_kpis_macro.csv
canal_ids  = [1,    2,    3,    4]
pesos_canal= [0.08, 0.25, 0.20, 0.47]

# Eventos comerciales argentinos — boost de ventas en estas fechas
# Generamos fechas entre 2022-01-01 y 2024-12-31
fecha_inicio = datetime(2022, 1, 1)
fecha_fin    = datetime(2024, 12, 31)
rango_dias   = (fecha_fin - fecha_inicio).days

def evento_comercial(fecha):
    """Devuelve el nombre del evento comercial si la fecha cae en uno, o vacío."""
    mes, dia = fecha.month, fecha.day
    if mes == 5  and 19 <= dia <= 21: return 'Hot Sale'
    if mes == 10 and 28 <= dia <= 30: return 'CyberMonday'
    if mes == 6  and dia == 20:       return 'Día del Padre'
    if mes == 10 and dia == 19:       return 'Día de la Madre'
    if mes == 12 and 20 <= dia <= 24: return 'Navidad'
    if mes == 2  and 1  <= dia <= 15: return 'Vuelta al Cole'
    return ''

print('Generando 150.000 transacciones... (puede tardar 30-60 segundos)')

filas_fact = []
for i in range(1, N_VENTAS + 1):
    
    # Cliente y canal
    cliente_id = random.randint(1, N_CLIENTES)
    canal_id   = np.random.choice(canal_ids, p=pesos_canal)
    
    # Zona del cliente (para plazo de entrega)
    zona_cliente = dim_clientes.loc[dim_clientes['ClienteID'] == cliente_id, 'zona'].values
    zona = zona_cliente[0] if len(zona_cliente) > 0 else 'Interior'
    
    # Fecha de la transacción
    dias_random = random.randint(0, rango_dias)
    fecha = fecha_inicio + timedelta(days=dias_random)
    
    # Medio de pago
    medio_pago = np.random.choice(medios_pago, p=pesos_pago)
    
    # Cuotas (solo aplica a tarjetas de crédito)
    if 'credito' in medio_pago:
        nro_cuotas = np.random.choice(cuotas_opts, p=pesos_cuotas)
    else:
        nro_cuotas = 1
    
    # Entrega (solo si es canal online)
    if canal_id == 1:  # físico
        tipo_entrega  = 'retiro_punto_venta'
        plazo_entrega = 'same_day'
    else:
        tipo_entrega  = np.random.choice(tipos_entrega, p=pesos_entrega)
        pesos_pl = pesos_AMBA if zona == 'AMBA' else pesos_interior
        plazo_entrega = np.random.choice(plazos, p=pesos_pl)
    
    # Sucursal (solo para canal físico)
    sucursal_id = random.randint(1, N_SUCURSALES) if canal_id == 1 else None
    
    filas_fact.append({
        'VentaID':          i,
        'ClienteID':        cliente_id,
        'CanalID':          canal_id,
        'SucursalID':       sucursal_id,
        'fecha':            fecha.strftime('%Y-%m-%d'),
        'anio':             fecha.year,
        'mes':              fecha.month,
        'trimestre':        (fecha.month - 1) // 3 + 1,
        'medio_pago':       medio_pago,
        'nro_cuotas':       nro_cuotas,
        'tipo_entrega':     tipo_entrega,
        'plazo_entrega':    plazo_entrega,
        'evento_comercial': evento_comercial(fecha),
        # Las columnas de monto, producto y margen se agregan en el ETL de Olist/Superstore
        'precio_venta_ars': None,
        'costo_unitario_ars': None,
        'cantidad': None,
        'ProductoID': None,
        'TiempoID': None,
    })
    
    if i % 30_000 == 0:
        print(f'  {i:,} / {N_VENTAS:,} generadas...')

fact_ventas_base = pd.DataFrame(filas_fact)
fact_ventas_base.to_csv(f'{OUTPUT_DIR}/005_fact_ventas_base_ar.csv', index=False, encoding='utf-8-sig')

print(f'\nFactVentas base generada: {len(fact_ventas_base):,} filas')
print('\nDistribución medio de pago:')
print(fact_ventas_base['medio_pago'].value_counts(normalize=True).round(3))
print('\nDistribución tipo entrega:')
print(fact_ventas_base['tipo_entrega'].value_counts(normalize=True).round(3))
print('\nEventos comerciales detectados:')
print(fact_ventas_base[fact_ventas_base['evento_comercial'] != '']['evento_comercial'].value_counts())

Generando 150.000 transacciones... (puede tardar 30-60 segundos)
  30,000 / 150,000 generadas...
  60,000 / 150,000 generadas...
  90,000 / 150,000 generadas...
  120,000 / 150,000 generadas...
  150,000 / 150,000 generadas...

FactVentas base generada: 150,000 filas

Distribución medio de pago:
medio_pago
tarjeta_credito_plataforma    0.549
billetera_electronica         0.242
debito_online                 0.091
tarjeta_credito_gateway       0.049
transferencia_bancaria        0.029
efectivo_rapipago             0.020
debito_al_retirar             0.010
credito_al_retirar            0.010
Name: proportion, dtype: float64

Distribución tipo entrega:
tipo_entrega
envio_domicilio              0.551
retiro_punto_venta           0.347
retiro_operador_logistico    0.074
retiro_pickup                0.019
mensajeria_rapida            0.009
Name: proportion, dtype: float64

Eventos comerciales detectados:
evento_comercial
Vuelta al Cole     6106
Navidad            2034
Hot Sale           1222


## BLOQUE 8 — FactPreciosComp (competencia)

Tabla de precios de competencia por SKU y período.  
Simula que el equipo comercial relevó precios del mercado cada mes.

Esta tabla alimenta el **Módulo 2 de pricing** en Power BI y se vincula  
a DimProducto a través de USERELATIONSHIP en DAX.

In [14]:
# Categorías con price_index típico de mercado argentino
# price_index > 1 = somos más caros que competencia
# price_index < 1 = somos más baratos
categorias_pricing = [
    # (categoria,              precio_base_ars, variacion_comp_pct, volatilidad)
    ('Electrónica',            85_000,          0.05,               0.12),
    ('Electrodomésticos',      120_000,          0.03,               0.08),
    ('Indumentaria',           25_000,          -0.05,              0.15),
    ('Deportes',               18_000,           0.08,              0.10),
    ('Hogar y Deco',           35_000,           0.02,              0.09),
    ('Alimentos y Bebidas',     4_500,          -0.08,              0.06),
    ('Belleza',                 8_000,           0.01,              0.12),
    ('Construcción',           22_000,           0.06,              0.07),
    ('Accesorios Vehículos',   15_000,          -0.02,              0.11),
    ('Infantil',               12_000,           0.04,              0.13),
]

filas_precios = []
precio_id = 1

# Generar un relevamiento mensual por categoría y canal (2022-2024)
for anio in [2022, 2023, 2024]:
    for mes in range(1, 13):
        for cat, precio_base, variacion_comp, volatilidad in categorias_pricing:
            
            # Ajuste inflacionario mensual (inflación argentina ~5% mensual en 2023)
            factor_inflacion = 1 + (0.04 if anio == 2022 else 0.08 if anio == 2023 else 0.03)
            precio_ajustado  = precio_base * (factor_inflacion ** (mes / 12))
            
            # Precio propio con variación aleatoria
            precio_propio = precio_ajustado * (1 + np.random.normal(0, volatilidad))
            
            # Precio competencia
            precio_comp = precio_propio * (1 + variacion_comp + np.random.normal(0, volatilidad * 0.5))
            
            # Descuento aplicado
            descuento_pct = max(0, np.random.normal(0.08, 0.05))   # ~8% descuento promedio
            precio_web    = precio_propio * (1 - descuento_pct)
            
            filas_precios.append({
                'PrecioID':           precio_id,
                'categoria':          cat,
                'anio':               anio,
                'mes':                mes,
                'fecha_relevamiento': f'{anio}-{mes:02d}-15',
                'precio_lista_ars':   round(precio_propio, 2),
                'precio_web_ars':     round(precio_web, 2),
                'precio_competencia': round(precio_comp, 2),
                'descuento_pct':      round(descuento_pct, 4),
                'price_index':        round(precio_propio / precio_comp, 4),
                'CanalID':            random.choice([2, 3, 4]),   # solo canales online
            })
            precio_id += 1

fact_precios = pd.DataFrame(filas_precios)
fact_precios.to_csv(f'{OUTPUT_DIR}/006_fact_precios_comp.csv', index=False, encoding='utf-8-sig')

print(f'FactPreciosComp generada: {len(fact_precios):,} filas')
print('\nPrice index promedio por categoría (>1 = somos más caros):')
print(fact_precios.groupby('categoria')['price_index'].mean().round(3).to_string())

FactPreciosComp generada: 360 filas

Price index promedio por categoría (>1 = somos más caros):
categoria
Accesorios Vehículos    1.010
Alimentos y Bebidas     1.092
Belleza                 0.988
Construcción            0.943
Deportes                0.921
Electrodomésticos       0.976
Electrónica             0.946
Hogar y Deco            0.973
Indumentaria            1.054
Infantil                0.956


## BLOQUE 9 — Resumen y verificación final

In [16]:
import os

print('=' * 55)
print('RESUMEN — Archivos generados por este notebook')
print('=' * 55)

archivos = [
    ('001_dim_geografia_ar.csv',    'DimGeografia',    'Provincias y ciudades AR con pesos CACE'),
    ('002_dim_sucursales_ar.csv',   'DimSucursal',     '50 sucursales distribuidas por región'),
    ('003_dim_canal_ar.csv',        'DimCanal',        '4 canales con pesos de facturación'),
    ('004_dim_clientes_ar.csv',     'DimCliente',      '10.000 clientes con perfil CACE 2025'),
    ('005_fact_ventas_base_ar.csv', 'FactVentas base', '150.000 transacciones con cols AR'),
    ('006_fact_precios_comp.csv',   'FactPreciosComp', '360 relevamientos de precios por categoría'),
]

for archivo, tabla, desc in archivos:
    ruta = f'{OUTPUT_DIR}/{archivo}'
    if os.path.exists(ruta):
        size_kb = os.path.getsize(ruta) / 1024
        df_temp = pd.read_csv(ruta)
        print(f'✓ {tabla:<20} {len(df_temp):>8,} filas   {size_kb:>8.1f} KB   {desc}')
    else:
        print(f'✗ {tabla:<20} NO ENCONTRADO — reejecutá el bloque correspondiente')



RESUMEN — Archivos generados por este notebook
✓ DimGeografia               67 filas        3.2 KB   Provincias y ciudades AR con pesos CACE
✓ DimSucursal                50 filas        4.9 KB   50 sucursales distribuidas por región
✓ DimCanal                    4 filas        0.2 KB   4 canales con pesos de facturación
✓ DimCliente             10,000 filas     1412.7 KB   10.000 clientes con perfil CACE 2025
✓ FactVentas base       150,000 filas    13880.4 KB   150.000 transacciones con cols AR
✓ FactPreciosComp           360 filas       27.9 KB   360 relevamientos de precios por categoría
